# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

In [6]:
DATA_FRACTION = 1.0

## take out the column names so we don't do any calc's with them

In [7]:
citationHeader = rddCitations.first()
patentHeader = rddPatents.first()

rawCitations = rddCitations.filter(
    lambda line: line != citationHeader
)

rawPatents = rddPatents.filter(
    lambda line: line != patentHeader
)

In [8]:
rawCitations.take(3)

['3858241,956203', '3858241,1324234', '3858241,3398406']

In [9]:
rawPatents.take(3)

['3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,']

## Now change the citation data into KV pairs

In [10]:
citations = rawCitations.map(lambda line: line.split(",")).map(
    lambda fields: (int(fields[0]), int(fields[1])))

In [11]:
citations.take(3)

[(3858241, 956203), (3858241, 1324234), (3858241, 3398406)]

## Next, for every patent, make PATENT the key, and (OG_ROW, POSTATE) as the value

In [12]:
#AI-written parsing function for this
def parsePatent(line):
    fields = line.split(",")

    patent = int(fields[0])
    postate = fields[5].strip('"') #don't let quotes get in

    return (patent, (line, postate))

In [13]:
patents = rawPatents.map(parsePatent)
patents = patents.cache()

In [14]:
patents.take(3)

[(3070801, ('3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,', '')),
 (3070802, ('3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,', 'TX')),
 (3070803,
  ('3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,', 'IL'))]

## Now only use the data fraction we want.

In [15]:
if DATA_FRACTION < 1.0:
    workingPatents = patents.sample(
        False,
        DATA_FRACTION,
        seed=42
    )
else:
    workingPatents = patents

In [16]:
workingPatents = workingPatents.cache()

In [17]:
print("All patents:", patents.count())
print("Working patents:", workingPatents.count())

All patents: 2923922
Working patents: 2923922


## Now join the patents with the "citing"'s

In [18]:
patent_to_citing = workingPatents.join(citations).cache()

In [19]:
patent_to_citing.take(5)

[(3879702,
  (('3879702,1975,5590,1973,"US","NY",513055,2,2,367,2,21,7,10,0.8571,0.58,0.6111,10.6,6.8571,0,0,0,0',
    'NY'),
   3061795)),
 (3879702,
  (('3879702,1975,5590,1973,"US","NY",513055,2,2,367,2,21,7,10,0.8571,0.58,0.6111,10.6,6.8571,0,0,0,0',
    'NY'),
   3185939)),
 (3879702,
  (('3879702,1975,5590,1973,"US","NY",513055,2,2,367,2,21,7,10,0.8571,0.58,0.6111,10.6,6.8571,0,0,0,0',
    'NY'),
   3307118)),
 (3879702,
  (('3879702,1975,5590,1973,"US","NY",513055,2,2,367,2,21,7,10,0.8571,0.58,0.6111,10.6,6.8571,0,0,0,0',
    'NY'),
   3384764)),
 (3879702,
  (('3879702,1975,5590,1973,"US","NY",513055,2,2,367,2,21,7,10,0.8571,0.58,0.6111,10.6,6.8571,0,0,0,0',
    'NY'),
   3517226))]

## Now have to change the key to CITED so we can attach states to this

In [20]:
by_cited_patent = patent_to_citing.map(
    lambda x: (
        x[1][1],
        (
            x[0],
            x[1][0][1]
        )
    )
)

In [21]:
by_cited_patent.take(5)

[(3061795, (3879702, 'NY')),
 (3185939, (3879702, 'NY')),
 (3307118, (3879702, 'NY')),
 (3384764, (3879702, 'NY')),
 (3517226, (3879702, 'NY'))]

## Make a patent -> state lookup so I can join this with the by_cited

In [22]:
patent_states = patents.map(
    lambda x: (x[0], x[1][1])
)

In [23]:
patent_states.take(5)

[(3070801, ''),
 (3070802, 'TX'),
 (3070803, 'IL'),
 (3070804, 'OH'),
 (3070805, 'CA')]

In [24]:
patents_to_cited = by_cited_patent.join(
    patent_states
).cache()

In [25]:
patents_to_cited.take(5)

[(4299687, ((4650564, 'PA'), 'KY')),
 (4299687, ((5171424, 'KY'), 'KY')),
 (4299687, ((5215720, 'IL'), 'KY')),
 (4299687, ((4859424, 'IL'), 'KY')),
 (4299687, ((4384948, 'KY'), 'KY'))]

## Now filter to just costate citations

In [26]:
costate_citations = patents_to_cited.filter(
    lambda x:
        x[1][0][1] != "" and
        x[1][0][1] == x[1][1]
)

In [27]:
costate_citations.take(10)

[(4299687, ((5171424, 'KY'), 'KY')),
 (4299687, ((4384948, 'KY'), 'KY')),
 (4299687, ((4377470, 'KY'), 'KY')),
 (4299687, ((5045176, 'KY'), 'KY')),
 (4299687, ((5028272, 'KY'), 'KY')),
 (4299687, ((4390415, 'KY'), 'KY')),
 (4299687, ((4406773, 'KY'), 'KY')),
 (4299687, ((4419223, 'KY'), 'KY')),
 (4299687, ((4744883, 'KY'), 'KY')),
 (4299687, ((4576709, 'KY'), 'KY'))]

## Convert back to CITING CITED

In [28]:
costate_pairs = costate_citations.map(
    lambda x: (
        x[1][0][0],
        x[0]
    )
)

In [29]:
costate_pairs.take(10)

[(5171424, 4299687),
 (4384948, 4299687),
 (4377470, 4299687),
 (5045176, 4299687),
 (5028272, 4299687),
 (4390415, 4299687),
 (4406773, 4299687),
 (4419223, 4299687),
 (4744883, 4299687),
 (4576709, 4299687)]

## Now filter out duplicates

In [30]:
distinct_costate_pairs = costate_pairs.distinct()
costate_ones = distinct_costate_pairs.map(
    lambda x: (x[0], 1)
)

In [31]:
costate_counts = costate_ones.reduceByKey(
    operator.add
).cache()

In [32]:
costate_counts.take(10)

[(5171424, 11),
 (4542528, 4),
 (5969868, 6),
 (5015400, 6),
 (5358564, 4),
 (5071890, 2),
 (4940091, 3),
 (5086089, 8),
 (5588922, 12),
 (5030787, 34)]

## Order it to compare with SQL

In [33]:
costate_counts.takeOrdered(
    13,
    key=lambda x: -x[1]
)

[(5959466, 125),
 (5983822, 103),
 (6008204, 100),
 (5952345, 98),
 (5958954, 96),
 (5998655, 96),
 (5936426, 94),
 (5913855, 90),
 (5925042, 90),
 (5951547, 90),
 (5739256, 90),
 (5978329, 90),
 (5980517, 90)]

## Now need to augment original patent rows

In [34]:
patents_with_counts = workingPatents.leftOuterJoin(
    costate_counts
)

In [35]:
patents_with_counts.take(5)

[(3070840,
  (('3070840,1963,1096,,"GB","",,3,,425,5,51,,6,,0.7778,,,,,,,', ''), None)),
 (3070972,
  (('3070972,1963,1096,,"US","KY",,2,,62,6,69,,9,,0.3457,,,,,,,', 'KY'),
   None)),
 (3071008,
  (('3071008,1963,1096,,"US","CA",,2,,73,4,43,,7,,0.449,,,,,,,', 'CA'), None)),
 (3071344,
  (('3071344,1963,1096,,"US","WA",,1,,251,5,53,,9,,0.4938,,,,,,,', 'WA'),
   None)),
 (3071380,
  (('3071380,1963,1096,,"US","NJ",,2,,369,2,24,,0,,,,,,,,,', 'NJ'), None))]

## Now need to replace none with 0

In [36]:
def appendCostateCount(entry):
    patent = entry[0]
    patentData = entry[1][0]
    count = entry[1][1]

    originalLine = patentData[0]

    if count is None:
        count = 0

    return originalLine + "," + str(count)

In [37]:
augmented_patents = patents_with_counts.map(
    appendCostateCount
)

In [38]:
augmented_patents.take(5)

['3070840,1963,1096,,"GB","",,3,,425,5,51,,6,,0.7778,,,,,,,,0',
 '3070972,1963,1096,,"US","KY",,2,,62,6,69,,9,,0.3457,,,,,,,,0',
 '3071008,1963,1096,,"US","CA",,2,,73,4,43,,7,,0.449,,,,,,,,0',
 '3071344,1963,1096,,"US","WA",,1,,251,5,53,,9,,0.4938,,,,,,,,0',
 '3071380,1963,1096,,"US","NJ",,2,,369,2,24,,0,,,,,,,,,,0']

## Bring back the header

In [39]:
patent_header = rddPatents.first()
new_header = patent_header + ",COCITED_COUNT"
print(new_header)

"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD",COCITED_COUNT


In [40]:
output = sc.parallelize([new_header]).union(
    augmented_patents
)

In [42]:
output.saveAsTextFile(
    "rdd_augmented_patents2"
)